In [ ]:
# %%bash

# pip uninstall torch torchvision torchaudio -y -q

# pip install torch==2.3.1+cu118 torchvision==0.18.1+cu118 \
# --index-url https://download.pytorch.org/whl/cu118 -q

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability())

True
Tesla T4
(7, 5)


In [2]:
%%bash

wget "https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/pub.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/priv.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/model.pt" -q

In [3]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import requests
import random
import argparse

from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet18
import torchvision.transforms as transforms

from sklearn.metrics import roc_curve

In [4]:
# config for cluster file
# BASE = Path(__file__).parent
# PUB_PATH = BASE / "pub.pt"
# PRIV_PATH = BASE / "priv.pt"
# MODEL_PATH = BASE / "model.pt"
# OUTPUT_CSV = BASE / "submission.csv"

# config for online jupyter notebook
BASE = os.path.dirname(os.path.abspath("__file__"))
PUB_PATH = BASE + "/pub.pt"
PRIV_PATH = BASE + "/priv.pt"
MODEL_PATH = BASE + "/model.pt"
OUTPUT_CSV = BASE + "/submission.csv"

BASE_URL = "http://34.63.153.158"   #DONOT CHANGE
API_KEY = "team_IV 5cd96dee6700b068d36678d03d1b1cec"
TASK_ID = "01-mia"  #DONOT CHANGE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICES = True if torch.cuda.device_count() > 1 else False
# DEVICE = ("cuda:0", "cuda:1") if DEVICES else ("cuda", "cuda")

BATCH_SIZE = 64

In [5]:
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)

In [6]:
class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

In [7]:
# load datasets

def load_dataset(PUB_PATH = PUB_PATH, PRIV_PATH = PRIV_PATH):
    print("Loading datasets...")
    pub_ds = torch.load(PUB_PATH, weights_only=False)
    priv_ds = torch.load(PRIV_PATH, weights_only=False)

    # normalization (same as training)
    MEAN = [0.7406, 0.5331, 0.7059]
    STD = [0.1491, 0.1864, 0.1301]
    
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    
    pub_ds.transform = transform
    priv_ds.transform = transform

    return pub_ds, priv_ds

In [8]:
def make_loader(dataset, batch_size = 64, shuffle = False, device = DEVICE):
    def collate_fn(batch):
    # unwrap if needed
        if len(batch[0]) == 1:
            batch = [b[0] for b in batch]
    
        ids, imgs, labels, membership = zip(*batch)
    
        return (
            list(ids),
            torch.stack(imgs),
            torch.tensor(labels),
            torch.tensor(membership)
        )
    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle, collate_fn = collate_fn)

In [9]:
def load_model(MODEL_PATH = MODEL_PATH):
    # load model
    print("Loading model...")
    model = resnet18(weights=None)
    model.conv1 = torch.nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.fc = torch.nn.Linear(512, 9)
    
    # model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu")) # Template code
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
    model.eval()
    model.to(DEVICE)
    return model

In [10]:
def extract_outputs(model, loader, device = DEVICE):
    all_ids = []
    all_scores = []

    model.eval()

    with torch.no_grad():
        for ids, imgs, labels, _ in loader:
            imgs = imgs.to(device)

            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)

            scores = probs.max(dim=1).values

            all_ids.extend(ids)
            all_scores.extend(scores.cpu().tolist())

    return all_ids, all_scores

In [11]:
def write_submission(ids, scores, path = OUTPUT_CSV):
    print("Creating random submission...")

    df = pd.DataFrame({
        "id": ids,
        "score": scores
    })

    df.to_csv(path, index=False)
    print("Saved:", OUTPUT_CSV)

In [12]:
from collections import defaultdict

import torch.optim
import torch.nn.functional as F
from torch.utils.data import Subset, DataLoader

import pandas as pd
import os
import torch.nn as nn
import numpy as np
from torchvision.models import resnet18

def lira_attack(model, dataset, device, pub_test):

    id_, conf_pred = get_confidence(model, dataset, device)


    id_and_conf =  dict(zip(id_, conf_pred))

    stats_in = get_stats("conf_in.csv")
    stats_out = get_stats("conf_out.csv")

    id_and_scores =normalize_scores(compute_lira_scores(id_and_conf, stats_in, stats_out))   # = normalize_scores()
    for id_, score in id_and_scores.items():
        print(id_, score)

    print("length of the dictionary = ", len(id_and_scores))

    is_valid = validate_submission(id_and_scores, dataset)

    if pub_test:
        compute_tpr_at_fpr(id_and_scores, dataset)

    save_submission(id_and_scores)

# def create_conf_csv(dataset, device):
#     for i in range (2):
#         in_ds, out_ds = create_shadow_split(dataset)

#         in_model = train_model(in_ds, device, model_type = "IN")

#         ids, confs = get_confidence(in_model, in_ds, device)
#         save_conf("conf_in.csv", ids, confs)
#         ids, confs = get_confidence(in_model, out_ds, device)
#         save_conf("conf_out.csv", ids, confs)


#         out_model = train_model(out_ds, device, model_type = "OUT")

#         ids, confs = get_confidence(out_model, in_ds, device)
#         save_conf("conf_out.csv", ids, confs)
#         ids, confs = get_confidence(out_model,out_ds, device)
#         save_conf("conf_in.csv", ids, confs)

import threading
from concurrent.futures import ThreadPoolExecutor

def create_conf_csv(dataset, devices=("cuda:0", "cuda:1"), n_shadows=2):
    write_lock = threading.Lock()

    def run_shadow(i):
        in_ds, out_ds = create_shadow_split(dataset, seed=i)

        # Train both models concurrently on separate GPUs
        with ThreadPoolExecutor(max_workers=2) as ex:
            f_in  = ex.submit(train_model, in_ds,  devices[0], "IN")
            f_out = ex.submit(train_model, out_ds, devices[1], "OUT")
            in_model  = f_in.result()
            out_model = f_out.result()

        # Run all 4 confidence inferences concurrently
        # in_model on in_ds  → member conf    → conf_in.csv
        # in_model on out_ds → non-member conf → conf_out.csv
        # out_model on in_ds → non-member conf → conf_out.csv
        # out_model on out_ds→ member conf     → conf_in.csv
        with ThreadPoolExecutor(max_workers=4) as ex:
            f1 = ex.submit(get_confidence, in_model,  in_ds,  devices[0])
            f2 = ex.submit(get_confidence, in_model,  out_ds, devices[0])
            f3 = ex.submit(get_confidence, out_model, in_ds,  devices[1])
            f4 = ex.submit(get_confidence, out_model, out_ds, devices[1])
            ids1, confs1 = f1.result()
            ids2, confs2 = f2.result()
            ids3, confs3 = f3.result()
            ids4, confs4 = f4.result()

        with write_lock:
            save_conf("conf_in.csv",  ids1, confs1)
            save_conf("conf_out.csv", ids2, confs2)
            save_conf("conf_out.csv", ids3, confs3)
            save_conf("conf_in.csv",  ids4, confs4)

    for i in range(n_shadows):
        run_shadow(i)

def create_shadow_split(dataset, seed = None):
    if seed is not None:
        np.random.seed(seed)

    class_indices = defaultdict(list)

    #grouping by labels
    for i in range(len(dataset)):
        _, _, label = dataset[i][:3]
        class_indices[int(label)].append(i)

    in_idx = []
    out_idx = []

    for label, indices in class_indices.items():
        indices = np.array(indices)
        np.random.shuffle(indices)

        split = len(indices) // 2
        in_idx.extend(indices[:split])
        out_idx.extend(indices[split:])

    in_ds = Subset(dataset, in_idx)
    out_ds = Subset(dataset, out_idx)

    return in_ds, out_ds

def train_model(
        ds,
        device,
        model_type,
        epochs = 15,
        batch_size = 32,
        lr = 1e-3,
        weight_decay = 1e-5 ):

    model = create_shadow_model().to(device)

    loader = DataLoader(
        ds,
        batch_size = batch_size,
        shuffle = True,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in loader:
            ids, images, labels = batch[:3]

            images = images.to(device)
            labels = labels.to(device).long()

            optimizer.zero_grad()

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(ds)
        print(f"{model_type } [model : ]Epoch {epoch+1}/{epochs}, Loss: {avg_loss}")

    return model

def create_shadow_model():
    model = resnet18(weights = None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size = 3, stride = 1, padding = 1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 9)
    return model


def get_confidence(model, ds, device):
    loader = DataLoader(ds, batch_size = 32, shuffle = False)
    model = model.to(device)
    model.eval()

    ids_all = []
    confs_all = []

    with torch.no_grad():
        for batch in loader:
            ids, images, labels = batch[:3]
            images = images.to(device)
            labels = labels.to(device).long()

            logits = model(images)
            probs = F.softmax(logits, dim = 1)
            conf = probs[
                torch.arange(labels.size(0),
                device = device),
                labels
            ]
            conf = conf.clamp(1e-7, 1 - 1e-7)
            conf = torch.log(conf / (1 - conf))

            ids_all.extend(int(i) for i in ids)
            confs_all.extend(conf.cpu().detach().numpy())

    return ids_all, confs_all

def save_conf(file_name, ids, confs):
    df_new = pd.DataFrame({
        "id" : ids,
        "conf" : confs
    })

    if os.path.exists(file_name):
        df_old = pd.read_csv(file_name)
        df_all = pd.concat([df_old, df_new], ignore_index = True)
    else:
        df_all = df_new

    df_all.to_csv(file_name, index = False)




def get_stats(csv_file):
    df = pd.read_csv(csv_file)

    stats = {}

    for id_, group in df.groupby("id"):
        values = group["conf"].values


        mean = np.mean(values)
        std = np.std(values, ddof=1) if len(values) > 1 else 1e-6

        stats[id_] = {"mean" : mean, "std" : std}

    return stats

def gaussian_logpdf(x, mean, std):
    std = max(std, 1e-6)  # avoid division by zero
    return -0.5 * np.log(2 * np.pi * std**2) - ((x - mean)**2) / (2 * std**2)

def compute_lira_scores(id_and_conf, stats_in, stats_out):
    id_to_score = {}

    for id_, s in id_and_conf.items():

        # skip if stats missing
        if id_ not in stats_in or id_ not in stats_out:
            id_to_score[id_] = 0.0
            continue

        mu_in = stats_in[id_]["mean"]
        std_in = stats_in[id_]["std"]

        mu_out = stats_out[id_]["mean"]
        std_out = stats_out[id_]["std"]

        logp_in = gaussian_logpdf(s, mu_in, std_in)
        logp_out = gaussian_logpdf(s, mu_out, std_out)

        score = logp_in - logp_out

    # log likelihood ratio

        id_to_score[id_] = score

    return id_to_score




def compute_tpr_at_fpr(id_to_score, dataset, target_fpr=0.05):
    scores = []
    labels = []

    # collect scores + ground truth
    for sample in dataset:
        id_, _, _, membership = sample
        if id_ not in id_to_score:
            print("This id does not exists in the shadow_Conf : ", id_)
            continue

        scores.append(id_to_score[id_])
        labels.append(membership)

    scores = np.array(scores)
    labels = np.array(labels)

    # separate positives and negatives
    pos_scores = scores[labels == 1]  # members
    neg_scores = scores[labels == 0]  # non-members

    # threshold at top 5% of negative scores
    threshold = np.percentile(neg_scores, 100 * (1 - target_fpr))

    # TPR = fraction of positives above threshold
    tpr = np.mean(pos_scores >= threshold)

    print("TPR at 5% FPR : {}".format(tpr))



def normalize_scores(id_to_score):
    sorted_ids = sorted(id_to_score, key=lambda k: id_to_score[k])
    n = len(sorted_ids)
    return {id_: rank / (n - 1) for rank, id_ in enumerate(sorted_ids)}



def validate_submission(id_to_score, dataset):
    errors = []
    required_ids = set(sample[0] for sample in dataset)
    submitted_ids = list(id_to_score.keys())

    # Check for duplicates
    if len(submitted_ids) != len(set(submitted_ids)):
        errors.append("❌ Duplicate IDs found")
    else:
        print("✅ No duplicate IDs")

    # Check no missing IDs
    missing = required_ids - set(submitted_ids)
    extra = set(submitted_ids) - required_ids
    if missing:
        errors.append(f"❌ Missing IDs: {len(missing)} missing")
    else:
        print("✅ No missing IDs")
    if extra:
        errors.append(f"❌ Extra IDs not in dataset: {len(extra)} extra")
    else:
        print("✅ No extra IDs")

    # Check each sample appears exactly once
    if len(submitted_ids) == len(required_ids) and not missing and not extra:
        print("✅ Each sample appears exactly once")

    # Check scores are numeric and in [0, 1]
    invalid_scores = []
    for id_, score in id_to_score.items():
        if not isinstance(score, (int, float, np.floating)):
            invalid_scores.append((id_, score, "not numeric"))
        elif not (0 <= score <= 1):
            invalid_scores.append((id_, score, "out of range"))

    if invalid_scores:
        errors.append(f"❌ Invalid scores: {invalid_scores[:5]}...")  # show first 5
    else:
        print("✅ All scores are numeric and in [0, 1]")

    # Summary
    print("\n--- Validation Summary ---")
    if not errors:
        print("✅ All checks passed! Ready to submit.")
    else:
        for e in errors:
            print(e)

    return len(errors) == 0

def save_submission(id_and_scores, filename="submission.csv"):
    df = pd.DataFrame(list(id_and_scores.items()), columns=["id", "score"])
    df.to_csv(filename, index=False)  # overwrites by default

In [ ]:
# !rm -rf /kaggle/working/*.csv

IN [model : ]Epoch 8/15, Loss: 0.3928859752894504
OUT [model : ]Epoch 8/15, Loss: 0.3328908141699754
IN [model : ]Epoch 9/15, Loss: 0.3853473966932985
OUT [model : ]Epoch 9/15, Loss: 0.31834471144455595


In [13]:
for i in range(5):
    pub_ds, priv_ds = load_dataset()
    cleaned_priv_ds = TaskDataset()
    cleaned_priv_ds.ids = priv_ds.ids
    cleaned_priv_ds.labels = priv_ds.labels
    cleaned_priv_ds.imgs = priv_ds.imgs
    model = load_model()
    create_conf_csv(cleaned_priv_ds)
    lira_attack(model=model, dataset=cleaned_priv_ds, device=DEVICE, pub_test=False)

Loading datasets...
Loading model...
IN [model : ]Epoch 1/15, Loss: 1.1993457001925572
OUT [model : ]Epoch 1/15, Loss: 1.2626463799719707
OUT [model : ]Epoch 2/15, Loss: 0.9533944219312513
IN [model : ]Epoch 2/15, Loss: 0.910071444576155
OUT [model : ]Epoch 3/15, Loss: 0.8239303675954825
IN [model : ]Epoch 3/15, Loss: 0.7166029868917123
OUT [model : ]Epoch 4/15, Loss: 0.6614026812405377
IN [model : ]Epoch 4/15, Loss: 0.6589497863028209
OUT [model : ]Epoch 5/15, Loss: 0.5900891075198282
IN [model : ]Epoch 5/15, Loss: 0.5292017771394589
OUT [model : ]Epoch 6/15, Loss: 0.5419940938698331
IN [model : ]Epoch 6/15, Loss: 0.47726421310967404
OUT [model : ]Epoch 7/15, Loss: 0.49647754915489495
IN [model : ]Epoch 7/15, Loss: 0.4133478038921277
OUT [model : ]Epoch 8/15, Loss: 0.4054033730892425
IN [model : ]Epoch 8/15, Loss: 0.3953554460099208
OUT [model : ]Epoch 9/15, Loss: 0.37061806973194916
IN [model : ]Epoch 9/15, Loss: 0.3240632339619868
OUT [model : ]Epoch 10/15, Loss: 0.32793155826195197

In [1]:
!nvidia-smi

Wed Apr 29 13:04:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## NAVIE ATTACK

In [ ]:
def run_naive_attack(model, priv_data, device, batch_size=64):
    model.to(device)
    model.eval()

    def collate_fn(batch):
        ids, imgs, labels, _ = zip(*batch)
        return list(ids), torch.stack(imgs), torch.tensor(labels)

    loader = DataLoader(
        priv_data,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    all_ids = []
    all_scores = []

    with torch.no_grad():
        for id_, img, label in loader:
            img = img.to(device)

            logits = model(img)
            probs = F.softmax(logits, dim=1)

            scores = probs.max(dim=1).values

            all_ids.extend(id_)
            all_scores.extend(scores.cpu().tolist())

    df = pd.DataFrame({'id': all_ids, 'score': all_scores})
    df.to_csv(OUTPUT_CSV, index=False)
    print("Saved:", OUTPUT_CSV)

In [ ]:
run_naive_attack(model, priv_ds, device)

## LIRA ATTACK

In [ ]:
def compute_signals(model, loader, device = DEVICE):

    all_labels = []

    scores = {
        "max_conf": [],
        "neg_loss": [],
        "neg_entropy": [],
        "margin": []
    }

    with torch.no_grad():
        for _, imgs, labels, membership in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)

            # ---- SIGNALS ----

            # 1. max confidence
            max_conf = probs.max(dim=1).values

            # 2. negative loss
            loss = F.cross_entropy(logits, labels, reduction='none')
            neg_loss = -loss

            # 3. negative entropy
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)
            neg_entropy = -entropy

            # 4. margin (top1 - top2)
            top2 = torch.topk(probs, k=2, dim=1).values
            margin = top2[:, 0] - top2[:, 1]

            # ---- STORE ----
            scores["max_conf"].extend(max_conf.cpu().tolist())
            scores["neg_loss"].extend(neg_loss.cpu().tolist())
            scores["neg_entropy"].extend(neg_entropy.cpu().tolist())
            scores["margin"].extend(margin.cpu().tolist())

            all_labels.extend(membership.cpu().tolist())

    return scores, np.array(all_labels)

In [ ]:
def tpr_at_fpr(scores, labels, target_fpr=0.05):
    fpr, tpr, thresholds = roc_curve(labels, scores)

    # find last index where fpr <= target
    idx = np.where(fpr <= target_fpr)[0]

    if len(idx) == 0:
        return 0.0

    return tpr[idx[-1]]

In [21]:
def evaluate_baselines(model, pub_ds, device = DEVICE):

    scores_dict, labels = compute_signals(model, loader, device)

    results = {}

    print("\n=== Baseline Results (TPR @ 5% FPR) ===\n")

    for name, scores in scores_dict.items():
        tpr = tpr_at_fpr(scores, labels)
        results[name] = tpr
        print(f"{name:15s}: {tpr:.4f}")

    best = max(results, key=results.get)
    print(f"\nBest signal: {best} ({results[best]:.4f})")

    return results

In [ ]:
pub_ds, _ = load_dataset()
loader = make_loader(pub_ds)
model = load_model()
results = evaluate_baselines(model, pub_ds)

In [ ]:
def split_shadow(pub_ds, ratio=0.5):
    indices = np.arange(len(pub_ds))
    np.random.shuffle(indices)

    split = int(ratio * len(indices))

    train_idx = indices[:split]
    out_idx = indices[split:]

    shadow_train = Subset(pub_ds, train_idx)
    shadow_out = Subset(pub_ds, out_idx)

    return shadow_train, shadow_out

In [ ]:
def train_shadow(model, loader, device, epochs=3):
    model = model.to(device)
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for _, imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model

In [ ]:
def extract_features(model, loader, device):
    model.eval()

    X = []
    y = []

    with torch.no_grad():
        for _, imgs, labels, membership in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)

            # ---- features ----
            max_conf = probs.max(dim=1).values
            loss = F.cross_entropy(logits, labels, reduction='none')
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)

            feats = torch.stack([
                max_conf,
                -loss,          # IMPORTANT
                -entropy
            ], dim=1)

            X.append(feats.cpu())
            y.append(membership)

    X = torch.cat(X)
    y = torch.cat(y)

    return X, y

In [ ]:
def build_attack_dataset(shadow_model, train_loader, out_loader, device):
    X_in, y_in = extract_features(shadow_model, train_loader, device)
    X_out, y_out = extract_features(shadow_model, out_loader, device)

    X = torch.cat([X_in, X_out])
    y = torch.cat([y_in, y_out])

    return X, y

In [ ]:
# 1. split
pub_ds = load_dataset()
shadow_train_ds, shadow_out_ds = split_shadow(pub_ds)

# 2. loaders
train_loader = make_loader(shadow_train_ds, shuffle=True)
out_loader = make_loader(shadow_out_ds, shuffle=False)

# 3. shadow model
shadow_model = load_model()

# 4. train
shadow_model = train_shadow(shadow_model, train_loader, DEVICE)

# 5. build attack dataset
X_attack, y_attack = build_attack_dataset(
    shadow_model,
    train_loader,
    out_loader,
    DEVICE
)

print(X_attack.shape, y_attack.shape)

In [ ]:
# create random submission (remove this later or it will rewrite your actual submission)
print("Creating random submission...")
ids = [str(i) for i in priv_ds.ids]

df = pd.DataFrame({
    "id": ids,
    "score": [random.random() for _ in ids]
})

df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

In [ ]:
# # submit
# def die(msg):
#     print(msg, file=sys.stderr)
#     sys.exit(1)

# parser = argparse.ArgumentParser(description="Submit a CSV file to the server.")
# args = parser.parse_args()

# submit_path = OUTPUT_CSV

# if not submit_path.exists():
#     die(f"File not found: {submit_path}")

# try:
#     with open(submit_path, "rb") as f:
#         resp = requests.post(
#             f"{BASE_URL}/submit/{TASK_ID}",
#             headers={"X-API-Key": API_KEY},
#             files={"file": (submit_path.name, f, "application/csv")},
#             timeout=(10, 600),
#         )
#     try:
#         body = resp.json()
#     except Exception:
#         body = {"raw_text": resp.text}

#     if resp.status_code == 413:
#         die("Upload rejected: file too large (HTTP 413).")

#     resp.raise_for_status()

#     print("Successfully submitted.")
#     print("Server response:", body)
#     submission_id = body.get("submission_id")
#     if submission_id:
#         print(f"Submission ID: {submission_id}")

# except requests.exceptions.RequestException as e:
#     detail = getattr(e, "response", None)
#     print(f"Submission error: {e}")
#     if detail is not None:
#         try:
#             print("Server response:", detail.json())
#         except Exception:
#             print("Server response (text):", detail.text)
#     sys.exit(1)